<a href="https://colab.research.google.com/github/RaphaelRAY/projeto-bi-games/blob/main/dados_bq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dados *BigQuery*

## Módulos

In [ ]:
from google.colab import auth, drive
from google.cloud import bigquery

In [ ]:
import numpy as np
import pandas as pd
import os
import re

## Autenticar projeto

In [ ]:
auth.authenticate_user()
project_id = 'projetotestemaua5584'
client = bigquery.Client(project=project_id)

## Carregar Dados

### Dados do drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
drive_path = '/content/drive/MyDrive/dados_cartola/'

In [ ]:
all_files = []
for root, directories, files in os.walk(drive_path):
  for name in files:
    file_name = os.path.join(root, name)
    all_files.append(file_name)

In [ ]:
all_files

['/content/drive/MyDrive/dados_cartola/classificacao/classificacao_2014.csv',
 '/content/drive/MyDrive/dados_cartola/classificacao/classificacao_2015.csv',
 '/content/drive/MyDrive/dados_cartola/classificacao/classificacao_2016.csv',
 '/content/drive/MyDrive/dados_cartola/classificacao/classificacao_2017.csv',
 '/content/drive/MyDrive/dados_cartola/classificacao/classificacao_2018.csv',
 '/content/drive/MyDrive/dados_cartola/classificacao/classificacao_2019.csv',
 '/content/drive/MyDrive/dados_cartola/partidas/partidas_2014.csv',
 '/content/drive/MyDrive/dados_cartola/partidas/partidas_2015.csv',
 '/content/drive/MyDrive/dados_cartola/partidas/partidas_2016.csv',
 '/content/drive/MyDrive/dados_cartola/partidas/partidas_2017.csv',
 '/content/drive/MyDrive/dados_cartola/partidas/partidas_2018.csv',
 '/content/drive/MyDrive/dados_cartola/partidas/partidas_2019.csv',
 '/content/drive/MyDrive/dados_cartola/scouts/scouts_2014.csv',
 '/content/drive/MyDrive/dados_cartola/scouts/scouts_2015.cs

### Dados no *BigQuery*:

Entender como as tabelas são criadas no bq

In [ ]:
all_tables = []
for dataset_id in client.list_datasets(project_id):
  dataset_id = dataset_id.dataset_id
  tables = client.list_tables(dataset_id)
  for table in tables:
    tab = table.full_table_id
    all_tables.append(tab)

In [ ]:
all_tables

['projetotestemaua5584:TEMP.partidas_casa',
 'projetotestemaua5584:TEMP.partidas_fora',
 'projetotestemaua5584:cartola_classificacao.classificacao_2014',
 'projetotestemaua5584:cartola_classificacao.classificacao_2015',
 'projetotestemaua5584:cartola_classificacao.classificacao_2016',
 'projetotestemaua5584:cartola_classificacao.classificacao_2017',
 'projetotestemaua5584:cartola_classificacao.classificacao_2018',
 'projetotestemaua5584:cartola_partidas.partidas_2014',
 'projetotestemaua5584:cartola_partidas.partidas_2015',
 'projetotestemaua5584:cartola_partidas.partidas_2016',
 'projetotestemaua5584:cartola_scouts.scouts_2014',
 'projetotestemaua5584:cartola_scouts.scouts_2015']

### Carregar dados e adicionar no BQ

- Preparar os nomes das tabelas

In [ ]:
bq_tables = []
for root, directories, files in os.walk(drive_path):
  for file in files:
    table = re.sub('.csv', '', file)
    dataset = re.sub('[0-9]+.csv', '', file)
    dataset = re.sub('_', '.', dataset)
    bq_tab = 'cartola_'+dataset+table
    bq_tables.append(bq_tab)

In [ ]:
bq_tables

['cartola_classificacao.classificacao_2014',
 'cartola_classificacao.classificacao_2015',
 'cartola_classificacao.classificacao_2016',
 'cartola_classificacao.classificacao_2017',
 'cartola_classificacao.classificacao_2018',
 'cartola_classificacao.classificacao_2019',
 'cartola_partidas.partidas_2014',
 'cartola_partidas.partidas_2015',
 'cartola_partidas.partidas_2016',
 'cartola_partidas.partidas_2017',
 'cartola_partidas.partidas_2018',
 'cartola_partidas.partidas_2019',
 'cartola_scouts.scouts_2014',
 'cartola_scouts.scouts_2015',
 'cartola_scouts.scouts_2016',
 'cartola_scouts.scouts_2017',
 'cartola_scouts.scouts_2018',
 'cartola_scouts.scouts_2019']

- Adicionar os dados

In [ ]:
for file, table in zip(all_files, bq_tables):

  # carregar dados
  dados = pd.read_csv(file)

  # inserir no big query
  job = client.load_table_from_dataframe(dados, table,
                                         job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
  job.result()

/usr/local/lib/python3.7/dist-packages/google/cloud/bigquery/_pandas_helpers.py:275: UserWarning: Unable to determine type of column 'clube'.
  warnings.warn(u"Unable to determine type of column '{}'.".format(column))
/usr/local/lib/python3.7/dist-packages/google/cloud/bigquery/_pandas_helpers.py:275: UserWarning: Unable to determine type of column 'clube_casa'.
  warnings.warn(u"Unable to determine type of column '{}'.".format(column))
/usr/local/lib/python3.7/dist-packages/google/cloud/bigquery/_pandas_helpers.py:275: UserWarning: Unable to determine type of column 'apelido'.
  warnings.warn(u"Unable to determine type of column '{}'.".format(column))


## Buscar dados no BQ

- Criar consulta SQL

In [ ]:
dados_partidas_bq = client.query('''
  select *
  from `cartola_partidas.partidas_*` ''')

- Transformar resultado da consulta em um dataframe

In [ ]:
dados_partidas = dados_partidas_bq.to_dataframe()

In [ ]:
dados_partidas.head()

,rodada_id,clube_casa_id,clube_visitante_id,placar_oficial_mandante,placar_oficial_visitante,clube_casa,clube_visitante,ano,partidas_partida_data,partidas_local
0,12,341,284,0,0,CSA,Grêmio,2019,2019-07-29 20:00:00 UTC,Rei Pelé
1,18,341,315,2,0,CSA,Chapecoense,2019,2019-09-08 19:00:00 UTC,Rei Pelé
2,20,341,354,1,0,CSA,Ceará,2019,2019-09-22 16:00:00 UTC,Rei Pelé
3,24,341,285,1,0,CSA,Internacional,2019,2019-10-09 19:15:00 UTC,Rei Pelé
4,3,341,277,0,0,CSA,Santos,2019,2019-05-05 16:00:00 UTC,Rei Pelé
